In [56]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Jul_16_20:06:48_Pacific_Daylight_Time_2025
Cuda compilation tools, release 13.0, V13.0.48
Build cuda_13.0.r13.0/compiler.36260728_0


In [57]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
else:


    print("GPU is not available.")

GPU is available!


# Dataset

In [58]:
import pandas as pd

In [59]:
diagnoses = pd.read_csv('../dataset/mimic-iv-3.1/hosp/d_icd_diagnoses.csv')

In [60]:
diagnoses.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


### Filter out the diagnoses codes for cardio

In [61]:
filteredDiagnoses1 = diagnoses[diagnoses['long_title'].str.contains(r'\bAcute Myocardial Infarction\b', case=False, regex=True)]
filteredDiagnoses1.count()

icd_code       34
icd_version    34
long_title     34
dtype: int64

In [62]:
filteredDiagnoses1.to_csv('filtered_data/diagnoses1.csv', index=False)

In [63]:
filteredDiagnoses2 = diagnoses[diagnoses['icd_code'].astype(str).str.startswith('410')]
print(filteredDiagnoses2.count())

icd_code       30
icd_version    30
long_title     30
dtype: int64


In [64]:
filteredDiagnoses2.to_csv('filtered_data/diagnoses2.csv', index=False)

### Filter out diagnoses

In [65]:
diagnosesCodes = list(filteredDiagnoses2['icd_code'])
print(diagnosesCodes)

['41000', '41001', '41002', '41010', '41011', '41012', '41020', '41021', '41022', '41030', '41031', '41032', '41040', '41041', '41042', '41050', '41051', '41052', '41060', '41061', '41062', '41070', '41071', '41072', '41080', '41081', '41082', '41090', '41091', '41092']


In [66]:
all_diagnoses_df = pd.read_csv('../dataset/mimic-iv-3.1/hosp/diagnoses_icd.csv')
all_diagnoses_df

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
...,...,...,...,...,...
6364483,19999987,23865745,7,41401,9
6364484,19999987,23865745,8,78039,9
6364485,19999987,23865745,9,0413,9
6364486,19999987,23865745,10,36846,9


In [67]:
ami_admissions_df = all_diagnoses_df[all_diagnoses_df['icd_code'].isin(diagnosesCodes)]
ami_admissions_df.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
169,10000764,27897940,2,41071,9
417,10000980,26913865,1,41071,9
703,10001492,27463908,1,41071,9
1192,10002013,24760295,1,41071,9
1507,10002155,23822395,1,41011,9


In [68]:
ami_admissions_df.count()

subject_id     6794
hadm_id        6794
seq_num        6794
icd_code       6794
icd_version    6794
dtype: int64

In [69]:
ami_admissions_df.to_csv('../dataset/mimic-iv-3.1/ami_admissions.csv', index=False)

### Adding Features

In [70]:
ami_admissions_df = pd.read_csv('../dataset/mimic-iv-3.1/ami_admissions.csv')
icustays_df = pd.read_csv('../dataset/mimic-iv-3.1/icu/icustays.csv')

In [71]:
icustays_df.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [72]:
ami_admissions_df.count()
# icustays_df.count()

subject_id     6794
hadm_id        6794
seq_num        6794
icd_code       6794
icd_version    6794
dtype: int64

In [73]:
structured_df = pd.merge(ami_admissions_df, icustays_df, on=['hadm_id'], how='inner')

6,794 (The Initial Count): This was likely the number of unique hospital admissions (hadm_id) where a patient was diagnosed with Acute Myocardial Infarction. This is every AMI patient admitted to the hospital.

3,722 (The New Count): This is the result of merging that list with the icustays.csv.gz file. This number represents the subset of those AMI patients who were sick enough to be admitted to the ICU during their hospital stay.

Not every patient admitted to the hospital for a heart attack goes to the ICU. Some are treated on general cardiology wards, so they won't appear in the icustays table.

In [74]:
structured_df.count()

subject_id_x      3722
hadm_id           3722
seq_num           3722
icd_code          3722
icd_version       3722
subject_id_y      3722
stay_id           3722
first_careunit    3722
last_careunit     3722
intime            3722
outtime           3722
los               3722
dtype: int64

In [75]:
structured_df = structured_df.drop(columns=['subject_id_y'])

In [76]:
structured_df['intime'] = pd.to_datetime(structured_df['intime'])
structured_df['outtime'] = pd.to_datetime(structured_df['outtime'])

In [77]:
structured_df.head()

,subject_id_x,hadm_id,seq_num,icd_code,icd_version,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000980,26913865,1,41071,9,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
1,10002155,23822395,1,41011,9,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912
2,10005817,20626031,1,41071,9,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097
3,10006053,22942076,3,41071,9,32895909,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2111-11-13 23:40:00,2111-11-15 18:21:10,1.778588
4,10009686,29681222,1,41021,9,34139812,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2164-04-30 07:37:10,2164-05-01 18:19:40,1.446181


In [78]:
structured_df['stay_hours'] = (structured_df['outtime'] - structured_df['intime']).dt.total_seconds() / 3600
structured_df.head()

,subject_id_x,hadm_id,seq_num,icd_code,icd_version,stay_id,first_careunit,last_careunit,intime,outtime,los,stay_hours
0,10000980,26913865,1,41071,9,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,11.940833
1,10002155,23822395,1,41011,9,33685454,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2129-08-04 12:45:00,2129-08-10 17:02:38,6.178912,148.293889
2,10005817,20626031,1,41071,9,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,56.618333
3,10006053,22942076,3,41071,9,32895909,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2111-11-13 23:40:00,2111-11-15 18:21:10,1.778588,42.686111
4,10009686,29681222,1,41021,9,34139812,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2164-04-30 07:37:10,2164-05-01 18:19:40,1.446181,34.708333


In [83]:
# sort by intime
structured_df_sorted = structured_df.sort_values(by=['hadm_id', 'intime'])

# keep the first stays and remove the rest
structured_df_sorted = structured_df_sorted.drop_duplicates(subset='hadm_id', keep='first')

In [84]:
print("Count: " , structured_df['hadm_id'].count())
print("Unique id Count: " , structured_df['hadm_id'].nunique())
print("------------------------------------------------------------------")
print("Count: " , structured_df_sorted['hadm_id'].count())
print("Unique id Count: " , structured_df_sorted['hadm_id'].nunique())

Count:  3722
Unique id Count:  3247
------------------------------------------------------------------
Count:  3247
Unique id Count:  3247
